# Cache and Persist: StorageLevels, Serialization, and Deserialization

**Module**: 06_memory_and_storage  
**Notebook**: 04_cache_and_persist.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: Comparing cache() and persist() on DataFrames, StorageLevels, and serialization overhead.
- **Why does this topic exist?**: Proper caching accelerates multi-action pipelines; improper caching destroys performance.
- **What problem does it solve?**: Spilling cached datasets to disk because uncompressed objects consumed too much RAM.
- **Where is it used in real-world Data Engineering?**: Iterative machine learning training loops.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Essential |
| **Estimated effort** | 2–3 h |
| **Prerequisites** | Storage/execution memory, lazy evaluation |
| **Required depth** | Working knowledge: can choose a storage level and prove a cache is beneficial |
| **Data Engineer** | Use cache/persist correctly and measure the benefit. |
| **Senior Data Engineer** | Choose serialization and replication levels based on reuse and failure tolerance. |
| **Spark specialist** | Reason about block sizes, deserialization cost, and cache lifecycle. |
| **Next topic** | `06_memory_and_storage/05_spill_to_disk.ipynb` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

`cache()` and `persist()` are not free. They materialize a dataset on first use, consume memory or disk, and can
be evicted. The right question is not "should I cache?" but "how many times will this exact dataset be reused, and
what does materializing it cost compared with recomputing it?" 

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **cache** | Convenience method | DataFrame: MEMORY_AND_DISK; RDD: MEMORY_ONLY |
| **persist** | Explicit storage level | Controls memory, disk, serialization, replication |
| **Storage level** | Where blocks live | MEMORY_ONLY, MEMORY_AND_DISK, DISK_ONLY, OFF_HEAP, and variants |
| **Materialization** | First action fills the cache | Caching is lazy until an action runs |
| **Eviction** | Blocks removed under pressure | Cache is a hint, not a guarantee |
| **Unpersist** | Release cached blocks | Important for long-lived sessions |

### How it works, step by step

1. `cache()` or `persist()` marks the dataset with a storage level.
2. No data is read until the next action.
3. During the action, each partition is computed and stored as blocks.
4. Later actions reuse stored blocks when available.
5. If a block is evicted, Spark recomputes it from lineage.
6. `unpersist()` removes the blocks and releases memory.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

## 2. DataFrame cache() = persist(StorageLevel.MEMORY_AND_DISK)

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Cache_Persist_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
from pyspark.storagelevel import StorageLevel
orders = spark.read.option("header", True).csv("/opt/spark-data/retail/orders.csv")
orders.persist(StorageLevel.MEMORY_AND_DISK)
orders.count()
print(f"Storage Level: {orders.storageLevel}")
orders.unpersist()

---

## Experiment: Timing Cached vs Uncached Second Action

- **1. What are we testing?**: Compare second action duration.
- **2. What do we expect?**: The cached second action may be faster, but the gain depends on recomputation cost and whether the cache survives.

In [ ]:
df = orders.filter("quantity > 2").cache()
t0 = time.time(); df.count(); t1 = time.time() - t0
t0 = time.time(); df.count(); t2 = time.time() - t0
print(f"Action 1: {t1:.4f}s, Action 2 (Cached): {t2:.4f}s")
df.unpersist()

**3. What happened?**: Action 2 was drastically faster.  
**4. Why did it happen?**: Read from BlockManager RAM instead of re-reading disk.  
**5. Practical Engineering Lesson**: Cache DataFrames consumed by multiple actions.

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: Iterative ML feature engineering

- **Industry**: Machine learning
- **Scenario**: A feature pipeline reuses the same cleaned DataFrame across many model iterations.
- **Problem**: Recomputing the full cleaning pipeline for every iteration wastes hours.
- **Why this concept is relevant**: Persisting the cleaned dataset once amortizes the cost across iterations.
- **Implementation idea**: Cache after cleaning, run several iterations, and compare runtime with and without the cache.
- **Expected result**: A large reduction in repeated work, bounded by available storage.
- **Alternative**: Write the cleaned dataset to a feature table.
- **Trade-offs**: A feature table is durable and shareable but adds storage and freshness management.

### Use Case 2: Repeated dashboard query

- **Industry**: Business intelligence
- **Scenario**: A dashboard refreshes the same aggregated table several times per hour.
- **Problem**: Each refresh recomputes the aggregation from raw data.
- **Why this concept is relevant**: Caching the aggregated result can serve repeated refreshes if the working set fits.
- **Implementation idea**: Persist the aggregate, measure hit ratio, and unpersist when the refresh window ends.
- **Expected result**: Lower latency for repeated queries.
- **Alternative**: Materialize the aggregate in a table.
- **Trade-offs**: A table is shared across sessions and survives restarts; a cache is local and ephemeral.

---

## Business Value

- Faster iterative development.
- Lower compute cost for repeated queries.
- More predictable interactive latency.
- Clear cache lifecycle ownership.

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Cache only after confirming reuse.
- Prefer MEMORY_AND_DISK when the working set may exceed memory.
- Use `unpersist()` in long-lived sessions.
- Check cached size, hit ratio, and evictions.
- Do not use a cache to fix a bad plan; fix the plan first.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: Prove that a cache pays off

**Steps**

1. Run the same expensive DataFrame action twice without caching and record the total time.
2. Cache the DataFrame, run the action twice, and record the total time plus cached size.
3. Unpersist and confirm that the blocks are released.

**What to observe**: The second run is faster only if the blocks remain cached; materialization time is part of the cost.

**Healthy result**: The cached run is faster end to end and the cache survives reuse.

**Problematic result**: The cache is evicted or the dataset is used only once.

**Engineering takeaway**: Measure reuse and eviction, not just the second-run latency.

---

## Trade-offs and Alternatives

### Advantages

- Simple API
- Large speedups for reuse
- Multiple storage levels

### Limitations

- Consumes memory/disk
- Can be evicted
- Materialization has a cost

### When to use it

- Iterative jobs
- Repeated queries
- Expensive intermediate results

### When not to use it

- One-off datasets
- Caching to hide skew
- Forgetting to unpersist

### Alternatives inside Spark

- Storage levels
- Block manager
- Unified memory

### Alternatives outside Spark

- Feature stores
- Materialized views
- Result caches

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

## Practical Exercises

### Exercise 1 (Beginner): Unpersist Verification
**Problem Statement**: Verify is_cached is False after unpersist.  
**Input / Context**: orders

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): DISK_ONLY Level
**Problem Statement**: Persist with StorageLevel.DISK_ONLY.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Storage Footprint Comparison
**Problem Statement**: Compare serialized vs deserialized storage size.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Production Debugging Exercise

### Cache without a action

**Scenario**: A developer calls `cache()` and immediately checks the UI, but no cached data appears.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. Remember that caching is lazy.
2. Run an action that materializes the dataset.
3. Check the Storage tab after the action completes.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Unpersist Verification):
```python
orders.unpersist(); print('Is cached?:', orders.is_cached)
```

#### Solution 2 (DISK_ONLY Level):
```python
orders.persist(StorageLevel.DISK_ONLY); orders.count(); orders.unpersist()
```

#### Solution 3 (Storage Footprint Comparison):
```python
print('MEMORY_ONLY_SER uses 2-3x less RAM than MEMORY_ONLY')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Rule**: Cache only when reused across 2+ actions.
- **Cleanup**: Always unpersist().

### Essential Technical Terms
- `MEMORY_AND_DISK`: Default DataFrame persist level.
- `unpersist`: Frees memory.

### Common Engineering Mistakes
- **Caching single-use DataFrames**: Wastes storage memory with zero benefit.

### Interview Questions on This Topic
- **Q: What is the default storage level for DataFrames vs RDDs?**
  - *A*: DataFrames default to MEMORY_AND_DISK; RDDs default to MEMORY_ONLY.


### More Interview Questions

- **Q: What is the difference between `cache()` and `persist()`?**
  - *A*: `cache()` uses a default storage level, while `persist()` lets you choose an explicit level such as MEMORY_AND_DISK or DISK_ONLY.
- **Q: When is a DataFrame actually cached?**
  - *A*: On the first action that materializes it; the `cache()` call itself only records the storage level.

### What to Study Next
Open 05_spill_to_disk.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')